# ⚡ HUẤN LUYỆN MÔ HÌNH SIÊU NHẸ MiT-B0 CHO MOBILE (30,000 ITERS)

> **Kiến trúc mô hình:**
> * **Backbone:** SegFormer MiT-B0 siêu nhẹ (~3.7M params/stream, tổng ~25.8M params).
> * **3 Đóng góp SOTA:** Tri-stream (RGB + ELA + OCR Spatial) + Dual Prototype Contrastive Head + Focused Crop.
> * **Thời gian huấn luyện trên GPU A100:** ~2 tiếng 15 phút (3.6 – 3.8 iter/s).
> * **Dung lượng xuất ONNX INT8:** Siêu nhẹ ~25–35 MB (tốc độ ~1.5 giây / ô trên điện thoại).

### 🟢 BƯỚC 1: KẾT NỐI DRIVE & CÀI ĐẶT MÔI TRƯỜNG (~3 Phút)
> Nhấn chạy Cell này để kết nối Google Drive, tải mã nguồn TMD mới nhất và giải nén Dataset.

In [ ]:
# 1. Kết nối Google Drive
import os, sys, glob, shutil
from google.colab import drive
drive.mount("/content/drive")

# 2. Clone mã nguồn RTM mới nhất từ GitHub main branch
%cd /content
!rm -rf TMD
!git clone https://github.com/gagaduy/TMD.git

# 3. Khai báo đường dẫn ASCFormer vào PYTHONPATH
sys.path.insert(0, "/content/TMD/ASCFormer")
%env PYTHONPATH=/content/TMD/ASCFormer

# 4. Cài đặt các thư viện cần thiết
!pip install -q mmcv-lite==2.0.0 mmengine==0.7.0 timm einops prettytable rich onnx onnxruntime

# 5. Giải nén Dataset RealTextMan từ Google Drive sang Colab SSD
os.makedirs("/content/TMD/ASCFormer/data/ttd", exist_ok=True)
drive_zip = "/content/drive/MyDrive/RealTextMan.zip"
if not os.path.exists(drive_zip):
    matches = glob.glob("/content/drive/MyDrive/**/RealTextMan.zip", recursive=True)
    if matches:
        drive_zip = matches[0]
    else:
        raise FileNotFoundError("Không tìm thấy file RealTextMan.zip trên Google Drive của bạn!")

print(f"📦 Đang giải nén {drive_zip} sang ổ cứng SSD của Colab...")
!unzip -q -o "{drive_zip}" -d /content/TMD/ASCFormer/data/ttd/

# Kiểm tra tính toàn vẹn dataset
dataset_dir = "/content/TMD/ASCFormer/data/ttd/RealTextMan/JPEGImages"
assert os.path.exists(dataset_dir), f"Lỗi: Không tìm thấy thư mục {dataset_dir}"
print("🎉 BƯỚC 1 HOÀN TẤT: Môi trường và Dataset đã sẵn sàng 100%!")


### 🔥 BƯỚC 2: BẮT ĐẦU HUẤN LUYỆN MiT-B0 (30,000 BƯỚC) + TỰ ĐỘNG LƯU DRIVE
> * Tự động tải trước trọng số pretrained `mit_b0.pth`.
> * Checkpoint và file Log được lưu liên tục vào Google Drive (`/content/drive/MyDrive/RTM_B0_30k_Mobile`).
> * Nếu Colab bị ngắt kết nối giữa chừng, bạn chỉ cần bấm chạy lại Cell này, nó sẽ **tự động `--resume` từ checkpoint gần nhất**!

In [ ]:
import os, sys, glob, json, torch

%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer
if "/content/TMD/ASCFormer" not in sys.path:
    sys.path.insert(0, "/content/TMD/ASCFormer")

# Tải trước trọng số Pretrained MiT-B0 vào cache
print("📥 Đang tải sẵn Pretrained SegFormer MiT-B0...")
_ = torch.hub.load_state_dict_from_url(
    "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth",
    map_location="cpu"
)

CFG_PATH = "configs/ascformer/ascformer_rtm_b0_30k_sota_mobile.py"
DRIVE_WORK_DIR = "/content/drive/MyDrive/RTM_B0_30k_Mobile"
os.makedirs(DRIVE_WORK_DIR, exist_ok=True)

# Kiểm tra Checkpoint cũ trên Drive để Resume nếu có
ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/iter_*.pth"), key=os.path.getmtime)
resume_flag = f"--resume {ckpts[-1]}" if ckpts else ""
if resume_flag:
    print(f"🔄 Tìm thấy checkpoint cũ trên Drive! Tự động tiếp tục từ: {ckpts[-1]}")
else:
    print("🚀 Bắt đầu huấn luyện MiT-B0 mới từ đầu (Iteration 0 -> 30,000)...")

# Chạy lệnh Train chuẩn
!python tools/train.py {CFG_PATH} --work-dir {DRIVE_WORK_DIR} {resume_flag}

print("🎉 HUẤN LUYỆN HOÀN TẤT! Checkpoint tốt nhất đã nằm trong Drive tại: " + DRIVE_WORK_DIR)


### 📦 BƯỚC 3: XUẤT MODEL ONNX INT8 SIÊU NHẸ SANG GOOGLE DRIVE
> Chạy Cell này để tự động nén mô hình sang định dạng ONNX INT8 siêu nhẹ để tích hợp vào Android App.

In [ ]:
import os, sys, glob

%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer
if "/content/TMD/ASCFormer" not in sys.path:
    sys.path.insert(0, "/content/TMD/ASCFormer")

DRIVE_WORK_DIR = "/content/drive/MyDrive/RTM_B0_30k_Mobile"
best_ckpts = sorted(glob.glob(f"{DRIVE_WORK_DIR}/*best*.pth") + glob.glob(f"{DRIVE_WORK_DIR}/iter_*.pth"), key=os.path.getmtime)
if not best_ckpts:
    raise FileNotFoundError("Chưa có checkpoint nào trong Drive!")
best_ckpt = best_ckpts[-1]
print(f"🎯 Đang sử dụng checkpoint tốt nhất: {best_ckpt}")

# Xuất mô hình sang ONNX INT8
out_onnx = f"{DRIVE_WORK_DIR}/rtm_b0_int8.onnx"
!python tools/export_ondevice_model.py \
    --config configs/ascformer/ascformer_rtm_b0_30k_sota_mobile.py \
    --checkpoint "{best_ckpt}" \
    --output "{out_onnx}"

if os.path.exists(out_onnx):
    sz_mb = os.path.getsize(out_onnx) / (1024 * 1024)
    print(f"✅ ĐÃ XUẤT THÀNH CÔNG MODEL ONNX SIÊU NHẸ: {out_onnx} ({sz_mb:.1f} MB)")
    print("📲 File đã lưu an toàn trên Google Drive của bạn! Bạn có thể tải về máy hoặc nạp vào APK.")
